In [ ]:
!pip install -q huggingface_hub datasets

In [ ]:
from huggingface_hub import login, whoami
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=token)
print(whoami()["name"])

In [ ]:
from pathlib import Path
import json
import os

repo_id = "nvidia/PhysicalAI-Robotics-GR00T-X-Embodiment-Sim"
local_dir = Path("/kaggle/working/gr00t_x_embodiment_sim")
report_path = Path("/kaggle/working/download_report_1_5.json")

# Subsets 1-5 in the GR-1 priority list from the CK plan.
SUBSETS_1_5 = [
    "gr1_arms_only.CanSort",
]

SUBSETS = SUBSETS_1_5

def folder_size_gb(path):
    path = Path(path)
    if not path.exists():
        return 0.0
    total = sum(p.stat().st_size for p in path.rglob("*") if p.is_file())
    return total / (1024 ** 3)

print("Target subsets:")
for i, subset in enumerate(SUBSETS, start=1):
    print(f"  {i}. {subset}")

print(f"\nLocal output: {local_dir}")
print(f"Report output: {report_path}")

In [ ]:
from huggingface_hub import snapshot_download
import time

local_dir.mkdir(parents=True, exist_ok=True)

for subset_name in SUBSETS:
    subset_path = local_dir / subset_name
    print("\n" + "=" * 80)
    print(f"Downloading subset: {subset_name}")

    snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        allow_patterns=f"{subset_name}/**",
        local_dir=str(local_dir),
        max_workers=3,
    )

    print(f"Downloaded to: {subset_path}")
    print(f"Subset size: {folder_size_gb(subset_path):.2f} GB")
    time.sleep(15)

print("\nDownload step finished.")

In [ ]:
report = {
    "repo_id": repo_id,
    "local_dir": str(local_dir),
    "subsets": [],
}

for subset_name in SUBSETS:
    subset_path = local_dir / subset_name
    data_dir = subset_path / "data"
    videos_dir = subset_path / "videos"
    meta_dir = subset_path / "meta"

    item = {
        "name": subset_name,
        "path": str(subset_path),
        "exists": subset_path.exists(),
        "size_gb": round(folder_size_gb(subset_path), 3),
        "has_data_dir": data_dir.exists(),
        "has_videos_dir": videos_dir.exists(),
        "has_meta_dir": meta_dir.exists(),
        "num_parquet_files": len(list(data_dir.rglob("*.parquet"))) if data_dir.exists() else 0,
        "num_video_files": len(list(videos_dir.rglob("*.mp4"))) if videos_dir.exists() else 0,
    }
    report["subsets"].append(item)

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))
print("\nDisk usage:")
os.system("df -h /kaggle/working")
os.system("du -sh /kaggle/working/gr00t_x_embodiment_sim 2>/dev/null || true")
print(f"\nSaved report to: {report_path}")